In [43]:
%%capture
!pip install tqdm pandas numpy scikit-learn nltk llm bertopic sentence-transformers cuml accelerate

In [1]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
import llm

from sentence_transformers import SentenceTransformer
import torch
from bertopic import BERTopic
from bertopic.representation import BaseRepresentation
from bertopic.vectorizers import ClassTfidfTransformer
try:
    from cuml.manifold import UMAP
    from cuml.cluster import HDBSCAN
except ImportError:
    from umap import UMAP
    from hdbscan import HDBSCAN

from const import DEFAULT_CHAT_PROMPT, DEFAULT_SYSTEM_PROMPT

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Overview

After extracting paragraphs from each available speech, we can compute embeddings and use them to perform a semi-supervised clustering. 
* The semi-supervised comes from the usage of the 21 topics.

Which embedding model to use? 
* Many of them are available, I checked [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) and selected the 16th best multilingual model (Qwen3-Embedding-0.6B). Most of the model in the MTEB Leaderboard are supported in SentenceTransformers and can be used to compute embeddings later used inside BERTopic library.

We pre-compute all the embeddings for the paragraphs and store them locally.
Then we need to define some hyperparameters for the BERTopic library, specifically UMAP and HDBSCAN configurations. No specific hyperparameter search was applied, just tuned them a bit manually.

After configuring the BERTopic instance, we can fit the model with precomputed embeddings and documents (which in our case correspond to paragraphs where each paragraph comes from a given speech).

**The main issue identified so far is that HDBSCAN tends to put the majority of the paragraphs inside the noise cluster (ID: -1)**. Right after fitting, ~220k paragraphs fall into such cluster. One way of reducing outliers is to post-hoc refine the cluster assignment using again the embeddings. Given that Qwen embeddings should be very sharp and precise, we can split the noise cluster using the Qwen embeddings and see if we still have a large noise cluster at the end.

After running the outlier reduction, we get around 160 topics where some of them fall inside our predefined list of 21 topics while others are identified just by the most frequent words inside those paragraphs.

What we can do now is to assign to the original paragraphs dataframe the identified topic name and then aggregate all the topics for each paragraph at the speech level, thus creating a list of topics discussed in a given speech by different MPs.

**Missing steps**: many of the identified topics are described by the most frequent terms, we need to refine such topic descriptions. The idea here is to use an LLM (Qwen) which takes as input the most frequent terms, the most relevant documents for those terms, and the list of 21 topics; prompt such model to first check whether one of the 21 topics can be assigned to such cluster of documents and, if not, propose a new human readable label that is representative of the documents.

In [2]:
df = pd.read_parquet("clean_paragraphs.parquet")
texts = df["TEXT_SPAN"].tolist()
df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name
0,81893,81893_p0,La commission propose de ne pas donner suite à...,President,FR,Bugnon André
1,81894,81894_p0,"Ich möchte Sie nur nochmals bitten, das Result...",For the committee,DE,Widmer Hans
2,81894,81894_p1,"In diesem Sinne bitte ich Sie nochmals, den An...",For the committee,DE,Widmer Hans
3,81895,81895_p0,Der Bundesrat beantragt die Annahme des Postul...,Member of the federal council,DE,Schmid Samuel
4,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel
...,...,...,...,...,...,...
303379,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine
303380,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine
303381,288681,288681_p13,En ce qui concerne le cas précis de cette prop...,For the committee,FR,Python Valentine
303382,288681,288681_p14,La majorité de la commission désire prendre en...,For the committee,FR,Python Valentine


In [3]:
topic_df = pd.read_csv("240102_bills_swissdomainbert2_predictions.csv")
topic_df

,bill_number,text,date,chamber,domain_id,source,language,time_span_decade,time_span_25y,domain_id_prediction_swissdomainbert2,domain_logits_swissdomainbert2,domain_name_prediction_swissdomainbert2,embeddings_swissdomainbert2,umap_swissdomainbert2
0,...,"Motion der Kommissionsminderheit (Renschier, ....",1987-01-01,Unknown,NaN,graph,DE,1980.0,1975.0,3,"[1.4315958196675638e-07, 8.955157682066783e-05...",Civil Rights,"[-0.6744987368583679, -0.5489828586578369, -0....","[3.0162856578826904, 6.312497615814209]"
1,0.10335,"195. Freymond, vom 26. Juni 1969.Ein ausländis...",1969-06-26,Nationalrat,NaN,graph,DE,1960.0,1950.0,14,"[3.644416892711888e-06, 3.176500104018487e-05,...","Technology, Communication","[0.3503628969192505, -0.17170414328575134, -0....","[6.8938212394714355, 5.891744136810303]"
2,00.001,Der Europarat beging 1999 sein 50-jähriges Bes...,2000-01-12,Ständerat,NaN,graph,DE,2000.0,2000.0,13,"[2.939555633929558e-05, 4.6187258703866974e-05...",International Affairs,"[0.4193879961967468, -0.16224949061870575, 0.2...","[13.971299171447754, -0.31970149278640747]"
3,00.003,In seinem Postulat vom 6. Oktober 1976 beauftr...,2000-01-19,Ständerat,13.0,graph,DE,2000.0,2000.0,13,"[2.9378930776147172e-05, 3.518968151183799e-05...",International Affairs,"[0.42163193225860596, -0.15665242075920105, 0....","[13.986701011657715, -0.2620127201080322]"
4,00.006,Bericht des Bundesrates über seine Geschäftsfü...,2000-03-09,Nationalrat,NaN,graph,DE,2000.0,2000.0,15,"[7.351007297984324e-06, 5.184900146559812e-06,...",Government Operations,"[-0.26835861802101135, -0.4732978343963623, 0....","[-1.9103219509124756, -2.2852351665496826]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89398,zu9711,"Kommission betreffend weiterer Ausbau derETH, ...",1967-08-16,Ständerat,NaN,graph,DE,1960.0,1950.0,7,"[0.019061243161559105, 0.0025766072794795036, ...",Environment,"[-0.12733963131904602, -0.44963544607162476, 0...","[1.1191364526748657, -3.445490598678589]"
89399,zu9765,"79. Kommission für das Arbeitsvertragsrecht,vo...",1969-04-30,Nationalrat,NaN,graph,DE,1960.0,1950.0,5,"[2.07011016755132e-05, 2.7762122044805437e-05,...",Labor,"[0.1266014575958252, 0.058757562190294266, -0....","[0.7178791165351868, 4.478944778442383]"
89400,zu9800,Voranschlag 1968. Nachtrag II. Botschaftund Be...,1968-05-15,Ständerat,NaN,graph,DE,1960.0,1950.0,19,"[0.0478166788816452, 0.0008331555873155594, 0....","Public Finances, Taxation","[-1.7563389539718628, -0.3660387098789215, 0.1...","[6.067897796630859, 10.824517250061035]"
89401,zu9898,"Kommission für die 7. Revision der AHV,vom 10....",1968-09-10,Nationalrat,NaN,graph,DE,1960.0,1950.0,12,"[3.667620660507964e-07, 1.309118033532286e-05,...",Social Welfare,"[-0.5014676451683044, -0.15047380328178406, 0....","[-2.119245767593384, -6.209519386291504]"


In [4]:
# Load the embedding model
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "sdpa",
    },
    tokenizer_kwargs={
        "padding_side": "left"
    },
)

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Loading weights: 100% 310/310 [00:00<00:00, 4444.91it/s]


In [5]:
#embeddings = embedder.encode(
#    texts,
#    normalize_embeddings=True,
#    batch_size=64,
#    show_progress_bar=True,
#)

In [6]:
#np.save(f"paragraph_embeddings_{EMBEDDING_MODEL.split("/")[-1]}.npy", embeddings)

In [7]:
# Load pre-computed embeddings
embeddings = np.load(f"paragraph_embeddings_{EMBEDDING_MODEL.split("/")[-1]}.npy")

# Normalize embeddings to unit length
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings.shape

(303384, 1024)

In [19]:
# Hyperparameters for BERTopic
RANDOM_SEED = 42
N_SAMPLES = -1
if N_SAMPLES == -1:
    N_SAMPLES = len(texts)

texts_sample = texts[:N_SAMPLES]
embeddings_sample = embeddings[:N_SAMPLES]

print(f"Using {len(texts_sample)} samples for topic modeling.")

bills_topics = topic_df['domain_name_prediction_swissdomainbert2'].unique().tolist()
print(f"Found {len(bills_topics)} unique topics: {bills_topics}")

# UMAP
N_NEIGHBORS = 20
N_COMPONENTS = 5
METRIC = "cosine"

# HDBSCAN
MIN_CLUSTER_SIZE = 150
MIN_SAMPLES = 10
CLUSTER_SELECTION_METHOD = "leaf" # "eom" (excess of mass) or "leaf"
CLUSTER_SELECTION_EPSILON = 0.1

# Zero-Shot topic
ZEROSHOT_MIN_SIMILARITY = 0.65 # Min cos similarity between speech embedding and predefined topic embedding to assign the topic to the speech

Using 303384 samples for topic modeling.
Found 21 unique topics: ['Civil Rights', 'Technology, Communication', 'International Affairs', 'Government Operations', 'Economy', 'Education, Research', 'Culture, Arts, Religion, Sport, Entertainment', 'Public Finances, Taxation', 'Agriculture', 'Social Welfare', 'Labor', 'Health', 'National Security, Armed Forces, Defense', 'Environment', 'Energy', '(Public) Transportation', 'Finances, Financial Sector', 'Spatial Planing, Civil Engineering', 'Migration', 'Parliamentary Procedure', 'Criminal Law']


In [20]:
# Download stopwords for German, French, and Italian
nltk.download("stopwords")

# Combine stopwords from German, French, and Italian
stop_words_multi = (
    stopwords.words("german")
    + stopwords.words("french")
    + stopwords.words("italian")
)

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    metric=METRIC,
    random_state=RANDOM_SEED
)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    prediction_data=True
)

# CountVectorizer with custom stop words (German, French, Italian)
vectorizer_model = CountVectorizer(
    stop_words=stop_words_multi,
)
# TF-IDF transformer that reduces the importance of frequent words across topics
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# Initialize BERTopic with the specified models and parameters
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True,
    zeroshot_topic_list=bills_topics,
    zeroshot_min_similarity=ZEROSHOT_MIN_SIMILARITY,
)

[nltk_data] Downloading package stopwords to /myhome/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
overwrite_model = True
if os.path.exists("bertopic_model_raw") and not overwrite_model:
    topic_model = BERTopic.load(
        "bertopic_model_raw",
        embedding_model=embedder,
    )
else:
    print("No model found")

No model found


In [22]:
# Fit model
topics, probs = topic_model.fit_transform(texts_sample, embeddings=embeddings_sample)

2026-04-13 11:13:13,659 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 11:13:27,642 - BERTopic - Dimensionality - Completed ✓
2026-04-13 11:13:27,647 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-04-13 11:13:29,960 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-04-13 11:13:42,940 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 11:15:38,002 - BERTopic - Cluster - Completed ✓
2026-04-13 11:15:38,003 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2026-04-13 11:15:38,861 - BERTopic - Zeroshot Step 2 - Completed ✓
2026-04-13 11:15:38,879 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 11:15:54,296 - BERTopic - Representation - Completed ✓


In [23]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,224661,-1_etats_projet_article_loi,"[etats, projet, article, loi, fait, frage, que...",[Zum Massnahmenpaket für eine verbesserte Inte...
1,0,1,Civil Rights,"[ordre, minorité, motion, , , , , , , ]",[Motion d'ordre de la minorité]
2,1,2,"Technology, Communication","[760, technologie, innovation, kommission, , ,...",[760 Kommission für Technologie und Innovation...
3,2,2,International Affairs,"[201, eidgenössisches, auswärtige, angelegenhe...",[201 Département fédéral des affaires étrangèr...
4,3,1,Government Operations,"[unités, administratives, budget, , , , , , , ]",[0. Budget des unités administratives]
...,...,...,...,...,...
157,156,173,156_einbürgerung_niederlassungsbewilligung_ein...,"[einbürgerung, niederlassungsbewilligung, einb...",[Der Initiant irrt sich aber doppelt: Er irrt ...
158,157,366,157_bauzonen_rpg_bauten_bauzone,"[bauzonen, rpg, bauten, bauzone, nichtbaugebie...","[Jetzt zur zweiten Problematik, bei der auch d..."
159,158,174,158_zweitwohnungen_wohnungen_touristisch_erstw...,"[zweitwohnungen, wohnungen, touristisch, erstw...",[Im vorliegenden Gesetzentwurf werden die Vorg...
160,159,315,159_verlagerungspolitik_güterverkehr_korridor_...,"[verlagerungspolitik, güterverkehr, korridor, ...",[Diesen Verlagerungsbericht behandelte auch un...


In [24]:
new_topics = topic_model.reduce_outliers(texts_sample, topics, strategy="embeddings", embeddings=embeddings_sample)
topic_model.update_topics(texts_sample, topics=new_topics)

2026-04-13 11:16:07,131 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [35]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,599,Civil Rights,"[absolutes, absolue, scrutin, blancs, majorité...",[Motion d'ordre de la minorité]
1,1,20,"Technology, Communication","[technologie, 0477, innovation, a2310, 760, kt...",[760 Kommission für Technologie und Innovation...
2,2,23,International Affairs,"[affaires, étrangères, dae, département, préfe...",[201 Département fédéral des affaires étrangèr...
3,3,15,Government Operations,"[unités, administratives, budget, indicateurs,...",[0. Budget des unités administratives]
4,4,26,Economy,"[économie, économique, secrétariat, petit, eta...","[704 Secrétariat d'Etat à l'économie, Bloc 1: ..."
...,...,...,...,...,...
156,156,1125,156_einbürgerung_integration_niederlassungsbew...,"[einbürgerung, integration, niederlassungsbewi...",[Der Initiant irrt sich aber doppelt: Er irrt ...
157,157,1384,157_bauzonen_bauten_ausserhalb_bauzone,"[bauzonen, bauten, ausserhalb, bauzone, raumpl...","[Jetzt zur zweiten Problematik, bei der auch d..."
158,158,432,158_zweitwohnungen_wohnungen_erstwohnungen_tou...,"[zweitwohnungen, wohnungen, erstwohnungen, tou...",[Im vorliegenden Gesetzentwurf werden die Vorg...
159,159,1889,159_verkehr_güterverkehr_den_auch,"[verkehr, güterverkehr, den, auch, wir, das, i...",[Diesen Verlagerungsbericht behandelte auch un...


In [30]:
#topic_model.visualize_topics()

In [31]:
#top_n = 20
#topic_model.visualize_heatmap(top_n_topics=top_n, title=f"Top {top_n} Topics Heatmap")

In [32]:
#topics_per_class = topic_model.topics_per_class(texts_sample, df.iloc[:N_SAMPLES]["name"])

In [33]:
#topic_model.visualize_topics_per_class(topics_per_class)

In [36]:
# Save model
topic_model.save(
    path=f"bertopic_model_{N_SAMPLES}_samples",
    serialization="safetensors",
    save_embedding_model=True,
    save_ctfidf=False,
)

### Qwen LLM topic refinement

In [ ]:
# class QwenTopic(BaseRepresentation):
#     def __init__(
#         self,
#         model: str = "sdsc-qwen-32b",
#         prompt=DEFAULT_CHAT_PROMPT,
#         system_prompt=DEFAULT_SYSTEM_PROMPT,
#         nr_docs: int = 4,
#         diversity: float | None = None,
#         **kwargs,
#     ):
#         """
#         Use Qwen to extract topic labels from a set of documents and keywords.

#         Arguments:
#             model: The Qwen model to use (e.g., "sdsc-qwen-32b")
#             prompt: The prompt template to use for Qwen. It should contain [KEYWORDS] and/or [DOCUMENTS] as placeholders.
#             system_prompt: The system prompt to set the behavior of the Qwen model.
#             nr_docs: The number of documents to pass if a prompt
#                  with the `["DOCUMENTS"]` tag is used.
#             diversity: Accepts values between 0 and 1. A higher
#                    values results in passing more diverse documents
#                    whereas lower values passes more similar documents.
#         """
#         self.model = model
#         self.prompt = prompt
#         self.system_prompt = system_prompt
#         self.nr_docs = nr_docs
#         self.diversity = diversity

#         self.prompts_ = []

#     def extract_topics(self, topic_model, documents, c_tf_idf, topics):
#         """Extract topics

#         Arguments:
#             topic_model: The BERTopic model
#             documents: A dataframe of documents with their related topics
#             c_tf_idf: The c-TF-IDF matrix
#             topics: The candidate topics as calculated with c-TF-IDF

#         Returns:
#             updated_topics: Updated topic representations
#         """
#         repr_docs_mappings, _, _, _ = topic_model._extract_representative_docs(
#             c_tf_idf, documents, topics, 500, self.nr_docs, self.diversity
#         )

#         updated_topics = {}
#         for topic, docs in tqdm(
#             repr_docs_mappings.items(), disable=not topic_model.verbose
#         ):
#             prompt = self._create_prompt(docs, topic, topics)
#             self.prompts_.append(prompt)

#             qwen_model = llm.get_model(self.model)
#             response = qwen_model.prompt(prompt=prompt, system=self.system_prompt)
#             answer = response.text()
#             after_thinking = answer.split("</think>")[-1]
#             label = after_thinking.replace("topic:", "").strip()
#             updated_topics[topic] = [(label, 1)]

#         return updated_topics

#     def _create_prompt(self, docs, topic, topics):
#         keywords = next(zip(*topics[topic]))

#         # Use the Default Chat Prompt
#         if self.prompt == DEFAULT_CHAT_PROMPT:
#             prompt = self.prompt.replace("[KEYWORDS]", ", ".join(keywords))
#             prompt = self._replace_documents(prompt, docs)

#         # Use a custom prompt that leverages keywords, documents or both using
#         # custom tags, namely [KEYWORDS] and [DOCUMENTS] respectively
#         else:
#             prompt = self.prompt
#             if "[KEYWORDS]" in prompt:
#                 prompt = prompt.replace("[KEYWORDS]", ", ".join(keywords))
#             if "[DOCUMENTS]" in prompt:
#                 prompt = self._replace_documents(prompt, docs)

#         return prompt

#     @staticmethod
#     def _replace_documents(prompt, docs):
#         to_replace = ""
#         for doc in docs:
#             to_replace += f"- {doc}\n"
#         prompt = prompt.replace("[DOCUMENTS]", to_replace)
#         return prompt

#     def get_prompts(self):
#         """Returns the prompts used for topic extraction."""
#         return self.prompts_


In [ ]:
llm_labels = {}
qwen_model = llm.get_model("sdsc-qwen-32b")

In [ ]:
for _, row in tqdm(valid_topics.iterrows(), total=len(valid_topics)):
    topic_id = row["Topic"]
    keywords = row["Representation"]
    rep_docs = row["Representative_Docs"][:3]

    prompt = DEFAULT_CHAT_PROMPT.replace("[KEYWORDS]", ", ".join(keywords))
    docs_text = "\n".join([f"- {doc}" for doc in rep_docs])
    prompt = prompt.replace("[DOCUMENTS]", docs_text)

    try:
        response = qwen_model.prompt(prompt=prompt, system=DEFAULT_SYSTEM_PROMPT)
        answer = response.text().split("</think>")[-1]
        label = answer.replace("topic:", "").strip()
        llm_labels[topic_id] = label
    except Exception as e:
        print(f"Failed on topic {topic_id}: {e}")
        llm_labels[topic_id] = "ERROR"
